# Model Training

## Purpose

This notebook trains and compares machine learning models for schedule-time flight-delay prediction using the engineered `flights_features` dataset.

The model-training workflow includes:

1. Load and validate flights_features
2. Split chronologically into train / validation / test
3. Engineer leakage-safe historical features
   - Historical airline delay rate
   - Historical origin-airport delay rate
   - Historical destination-airport delay rate
   - Historical route delay rate, if support is sufficient
   - Calculate mappings from training data only
   - Apply fallback global training rate to unseen categories
4. Encode categorical variables
5. Train multiple candidate models
6. Tune hyperparameters
7. Evaluate model performance
8. Select the best model
9. Save the selected model and supporting artifacts
10. Prepare model outputs for SHAP explainability

## 1. Load and Validate the Feature Dataset

The model-training process begins by loading the managed `flights_features` Delta table produced by the Feature Engineering notebook.

Before splitting or modelling, the dataset is validated to confirm that:

- The required Unity Catalog table exists
- The target variable is available
- The flight date is stored as a valid date
- All required schedule-time predictors are present
- The dataset contains records suitable for chronological splitting

In [0]:
from __future__ import annotations

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T


FEATURE_TABLE = "workspace.default.flights_features"
TARGET_COLUMN = "ARR_DEL15"
DATE_COLUMN = "FL_DATE"


def require_table(table_name: str) -> None:
    """Raise an error when a required Unity Catalog table is unavailable."""
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(
            f"Required table '{table_name}' was not found. "
            "Run the feature-engineering notebook before continuing."
        )


require_table(FEATURE_TABLE)

df_features: DataFrame = spark.table(FEATURE_TABLE)

required_columns = {
    "QUARTER",
    "MONTH",
    "DAY_OF_WEEK",
    "FL_DATE",
    "OP_UNIQUE_CARRIER",
    "ORIGIN",
    "DEST",
    "DISTANCE",
    "CRS_DEP_TIME",
    "CRS_ARR_TIME",
    "CRS_ELAPSED_TIME",
    "DEP_HOUR",
    "DEP_MINUTE",
    "IS_WEEKEND",
    "SEASON",
    "TIME_OF_DAY",
    "FLIGHT_DISTANCE_CATEGORY",
    "ARR_DEL15",
}

missing_columns = sorted(required_columns - set(df_features.columns))

if missing_columns:
    raise ValueError(
        "Model-training validation failed. "
        f"Missing required columns: {missing_columns}"
    )

feature_row_count = df_features.count()
feature_column_count = len(df_features.columns)

date_type = df_features.schema[DATE_COLUMN].dataType

if not isinstance(date_type, T.DateType):
    raise TypeError(
        f"{DATE_COLUMN} must be a Spark date column, "
        f"but found {date_type.simpleString()}."
    )

print("Feature dataset loaded and validated successfully.")
print(f"Source table: {FEATURE_TABLE}")
print(f"Total records: {feature_row_count:,}")
print(f"Total columns: {feature_column_count}")
print(f"Prediction target: {TARGET_COLUMN}")
print(f"Date column type: {date_type.simpleString()}")

### 1.1 Date Range and Target Distribution

Before defining the chronological training, validation, and test periods, the feature dataset is examined to confirm its available date range and the distribution of the binary target variable.

This review supports two important modelling decisions:

- Selecting non-overlapping chronological split periods
- Assessing whether the delayed and on-time classes are imbalanced

No records are modified during this analysis.

In [0]:
dataset_profile = (
    df_features
    .select(
        F.min("FL_DATE").alias("MIN_FL_DATE"),
        F.max("FL_DATE").alias("MAX_FL_DATE"),
        F.count("*").alias("TOTAL_RECORDS"),
        F.sum(
            F.when(F.col("ARR_DEL15") == 0, 1).otherwise(0)
        ).alias("ON_TIME_RECORDS"),
        F.sum(
            F.when(F.col("ARR_DEL15") == 1, 1).otherwise(0)
        ).alias("DELAYED_RECORDS"),
    )
    .withColumn(
        "ON_TIME_PERCENTAGE",
        F.round(
            F.col("ON_TIME_RECORDS") / F.col("TOTAL_RECORDS") * 100,
            4,
        ),
    )
    .withColumn(
        "DELAYED_PERCENTAGE",
        F.round(
            F.col("DELAYED_RECORDS") / F.col("TOTAL_RECORDS") * 100,
            4,
        ),
    )
)

display(dataset_profile)

## 2. Create the Chronological Train, Validation, and Test Split

The dataset is divided chronologically rather than randomly because the model is intended to predict future flight-delay risk from historical observations.

The split periods are defined as follows:

- **Training period:** January 1, 2025 to August 31, 2025
- **Validation period:** September 1, 2025 to October 31, 2025
- **Test period:** November 1, 2025 to December 31, 2025

This design ensures that later flight outcomes are not used to train models evaluated on earlier periods. The validation dataset will support model and hyperparameter selection, while the test dataset will remain untouched until final evaluation.

In [0]:
TRAIN_END_DATE = "2025-08-31"
VALIDATION_START_DATE = "2025-09-01"
VALIDATION_END_DATE = "2025-10-31"
TEST_START_DATE = "2025-11-01"

df_train = df_features.filter(
    F.col("FL_DATE") <= F.to_date(F.lit(TRAIN_END_DATE))
)

df_validation = df_features.filter(
    (F.col("FL_DATE") >= F.to_date(F.lit(VALIDATION_START_DATE)))
    & (F.col("FL_DATE") <= F.to_date(F.lit(VALIDATION_END_DATE)))
)

df_test = df_features.filter(
    F.col("FL_DATE") >= F.to_date(F.lit(TEST_START_DATE))
)

split_summary = (
    df_train.select(
        F.lit("TRAIN").alias("DATASET"),
        F.min("FL_DATE").alias("MIN_DATE"),
        F.max("FL_DATE").alias("MAX_DATE"),
        F.count("*").alias("TOTAL_RECORDS"),
        F.avg(F.col("ARR_DEL15").cast("double")).alias("DELAY_RATE"),
    )
    .unionByName(
        df_validation.select(
            F.lit("VALIDATION").alias("DATASET"),
            F.min("FL_DATE").alias("MIN_DATE"),
            F.max("FL_DATE").alias("MAX_DATE"),
            F.count("*").alias("TOTAL_RECORDS"),
            F.avg(F.col("ARR_DEL15").cast("double")).alias("DELAY_RATE"),
        )
    )
    .unionByName(
        df_test.select(
            F.lit("TEST").alias("DATASET"),
            F.min("FL_DATE").alias("MIN_DATE"),
            F.max("FL_DATE").alias("MAX_DATE"),
            F.count("*").alias("TOTAL_RECORDS"),
            F.avg(F.col("ARR_DEL15").cast("double")).alias("DELAY_RATE"),
        )
    )
    .withColumn(
        "DELAY_PERCENTAGE",
        F.round(F.col("DELAY_RATE") * 100, 4),
    )
    .drop("DELAY_RATE")
)

display(split_summary)

### Split Validation Summary

The chronological split produced three non-overlapping datasets whose combined record count matches the complete feature dataset.

The target distribution varies across the periods:

- The training period has a delay rate of approximately 22.91%.
- The validation period has a lower delay rate of approximately 18.55%.
- The test period has a higher delay rate of approximately 23.71%.

This variation reflects temporal changes in airline operations and confirms the importance of evaluating the model on future periods rather than using a random split.

## 3. Engineer Leakage-Safe Historical Features

Historical performance features summarize prior delay behaviour for airlines, airports, and routes.

To prevent target leakage:

- Historical features for training records use only flights from earlier dates.
- The current record and later training outcomes are excluded.
- Validation and test mappings will be calculated from the training period only.
- A global training delay rate will be used when insufficient historical observations are available.

The first feature created is `AIRLINE_HIST_DELAY_RATE`, representing an airline's smoothed arrival-delay rate before the current flight date.

In [0]:
from pyspark.sql.window import Window


# Daily airline-level delay statistics within the training period
airline_daily_stats = (
    df_train
    .groupBy(
        "OP_UNIQUE_CARRIER",
        "FL_DATE",
    )
    .agg(
        F.count("*").alias("DAILY_FLIGHT_COUNT"),
        F.sum(F.col("ARR_DEL15").cast("long")).alias("DAILY_DELAY_COUNT"),
    )
)

# Use only dates before the current flight date
airline_history_window = (
    Window
    .partitionBy("OP_UNIQUE_CARRIER")
    .orderBy(F.col("FL_DATE").cast("timestamp").cast("long"))
    .rowsBetween(Window.unboundedPreceding, -1)
)

airline_daily_history = (
    airline_daily_stats
    .withColumn(
        "AIRLINE_PRIOR_FLIGHTS",
        F.sum("DAILY_FLIGHT_COUNT").over(airline_history_window),
    )
    .withColumn(
        "AIRLINE_PRIOR_DELAYS",
        F.sum("DAILY_DELAY_COUNT").over(airline_history_window),
    )
)

display(
    airline_daily_history
    .select(
        "OP_UNIQUE_CARRIER",
        "FL_DATE",
        "DAILY_FLIGHT_COUNT",
        "DAILY_DELAY_COUNT",
        "AIRLINE_PRIOR_FLIGHTS",
        "AIRLINE_PRIOR_DELAYS",
    )
    .orderBy(
        "OP_UNIQUE_CARRIER",
        "FL_DATE",
    )
    .limit(30)
)

### 3.1 Historical Airline Delay Rate

The `AIRLINE_HIST_DELAY_RATE` feature represents an airline's arrival-delay rate using only flights from earlier training dates.

A smoothed estimate is used to prevent unstable rates when an airline has limited prior observations. The overall training delay rate serves as the prior and as the fallback value for the first available date, when no earlier airline history exists.

The current date's outcomes are excluded from the calculation.

In [0]:
# Overall delay rate from the training period.
# This is used as the smoothing prior and first-date fallback.
global_training_delay_rate = (
    df_train
    .select(F.avg(F.col("ARR_DEL15").cast("double")).alias("GLOBAL_DELAY_RATE"))
    .first()["GLOBAL_DELAY_RATE"]
)

# Controls how strongly low-volume airline histories are pulled
# toward the global training delay rate.
SMOOTHING_STRENGTH = 100.0

airline_history_features = (
    airline_daily_history
    .withColumn(
        "AIRLINE_HIST_DELAY_RATE",
        (
            F.coalesce(
                F.col("AIRLINE_PRIOR_DELAYS").cast("double"),
                F.lit(0.0),
            )
            + F.lit(SMOOTHING_STRENGTH * global_training_delay_rate)
        )
        /
        (
            F.coalesce(
                F.col("AIRLINE_PRIOR_FLIGHTS").cast("double"),
                F.lit(0.0),
            )
            + F.lit(SMOOTHING_STRENGTH)
        ),
    )
    .select(
        "OP_UNIQUE_CARRIER",
        "FL_DATE",
        "AIRLINE_PRIOR_FLIGHTS",
        "AIRLINE_PRIOR_DELAYS",
        "AIRLINE_HIST_DELAY_RATE",
    )
)

# Join the leakage-safe airline history to every training flight.
df_train_hist = (
    df_train
    .join(
        airline_history_features,
        on=["OP_UNIQUE_CARRIER", "FL_DATE"],
        how="left",
    )
    .withColumn(
        "AIRLINE_HIST_DELAY_RATE",
        F.coalesce(
            F.col("AIRLINE_HIST_DELAY_RATE"),
            F.lit(global_training_delay_rate),
        ),
    )
)

print(f"Global training delay rate: {global_training_delay_rate:.6f}")
print(f"Smoothing strength: {SMOOTHING_STRENGTH:.0f}")
print(f"Training rows after join: {df_train_hist.count():,}")

display(
    df_train_hist
    .select(
        "OP_UNIQUE_CARRIER",
        "FL_DATE",
        "AIRLINE_PRIOR_FLIGHTS",
        "AIRLINE_PRIOR_DELAYS",
        "AIRLINE_HIST_DELAY_RATE",
        "ARR_DEL15",
    )
    .orderBy(
        "OP_UNIQUE_CARRIER",
        "FL_DATE",
    )
    .limit(30)
)